<a href="https://colab.research.google.com/github/Shacxify/AAI2025/blob/main/03_self_reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q -U google-genai

In [2]:
from google import genai
from google.genai import types
import json, re

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-3.6-flash'

def ask(user_prompt, system=None, temperature=0.0, json_mode=False, model=MODEL):
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    if json_mode:
        kwargs['response_mime_type'] = 'application/json'
    resp = client.models.generate_content(
        model=model, contents=user_prompt,
        config=types.GenerateContentConfig(**kwargs),
    )
    return (resp.text or '').strip()

print('Connected. Using:', MODEL)

Connected. Using: gemini-2.5-flash


In [3]:
MEMO = """TO:      Store Ownership
FROM:    VNTG OS Platform Operations
RE:      Q3 FY26 consignor performance, intake throughput, and payout operations
DATE:    September 30, 2026

INTAKE VOLUME AND SLA
Q3 closed with 1,842 items accepted into intake across the 1st Street and Santana Row drop
points, up from 1,494 in Q2, a 23.3% increase quarter over quarter. Of those, 1,611 were
photographed, priced, and listed inside the ten business day intake SLA, putting on-time
listing at 87.5% against a target of 92%. The 231 late items averaged 14.2 business days from
drop-off to live listing. Late listings cluster almost entirely in the two weeks following each
of the three campus move-in weekends, when drop-off volume ran 41% above trailing average while
intake staffing stayed flat at two photographers per store.

SALES AND PAYOUTS
Gross merchandise value for the quarter was $96,412 across 1,203 sold items, an average sale
price of $80.14. Consignor payouts totaled $58,247, a blended consignor share of 60.4%,
consistent with the standard 60/40 split plus the 65% tier that applies to items priced at $200
and above. 87 items cleared the $200 tier this quarter, up from 54 in Q2.

SELL-THROUGH
Of the 1,611 items listed, 1,203 sold inside the quarter, a 74.7% sell-through rate. Items
priced under $45 sold at 81.2%, items between $45 and $120 at 76.9%, and items above $120 at
52.4%. Median days to sale was 19, up from 16 in Q2. The pricing team ran a limited test in
August, marking down 140 items that had sat past 45 days by 20%. 96 of those 140 cleared inside
three weeks, which is a 68.6% clearance rate on aged stock that had previously been moving at
roughly 22%.

RETURNS
68 items were returned inside the 7-day buyer return window, 5.7% of sold items, up from 4.1%
in Q2. 41 of the 68 return reasons cited fit or measurement discrepancy. Measurements are
entered by hand at intake and are not validated against anything. Each return also reverses a
payout line that has usually already been queued, which is the direct cause of the payout delays
described below.

CONSIGNOR RETENTION
312 active consignors at quarter end, 74 new this quarter, 38 churned, defined as no drop-off
within 90 days of their last payout. Churn concentrates hard among small first payouts: 26 of
the 38 churned consignors had a first payout under $25. Median first payout for retained
consignors was $61.

PAYOUT OPERATIONS
Payouts run twice monthly, on the 1st and the 15th. Three of the six Q3 cycles ran late, by an
average of 2.4 days. All three delays trace to manual reconciliation of returned items against
payout lines that were already queued. Ops estimates 6 to 8 staff hours per late cycle spent on
that reconciliation by hand.

SUPPORT LOAD
417 support tickets in Q3, 61% consignor-side. Top categories were intake status at 38%, payout
amount questions at 27%, and item condition disputes at 11%. Average first response time was
9.4 hours against a stated 8 hour target.

RECOMMENDATIONS
1. Add a third photographer at 1st Street for the four weeks following each move-in weekend.
   Estimated cost $3,100 per cycle; would have covered roughly 180 of the 231 late items.
2. Add measurement validation at intake, either a required second entry or a template per
   garment category. Targets the 41 fit-related returns.
3. Automate return reconciliation against the payout queue before the cycle runs. Removes the
   single named cause of all three late payout cycles and 6 to 8 hours of manual work each.
4. Test a first-payout floor or payout bundling for consignors under $25, aimed at the 26
   churned consignors in that band.
"""

print(f"source memo: {len(MEMO.split())} words")

source memo: 608 words


In [4]:
PROMPT_BEFORE = """Summarize the following memo for the business owner.

{memo}"""

summary_before = ask(PROMPT_BEFORE.format(memo=MEMO), temperature=0.3)
print(summary_before)
print('\n---')
print('word count:', len(summary_before.split()))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}

In [ ]:
REFLECT_SYSTEM = """You are a demanding editor reviewing a draft against a fixed checklist. You do
not praise. You do not soften. If a criterion passes, you say PASS and move on in one line."""

PROMPT_REFLECT = """Below are a source memo, a draft summary of it, and the five criteria the
summary must meet. Critique the draft, then rewrite it.

CRITERIA
1. FACTUAL ACCURACY - every claim traces to a specific line of the memo. No inference, no
   conclusions the memo does not state.
2. NUMERIC INTEGRITY - every number in the summary appears verbatim in the memo. Do not compute,
   combine, round, or annualize anything. If a number is not in the memo, it does not go in.
3. LENGTH - 120 words maximum, whitespace-delimited. Count before you submit.
4. FORMAT - exactly three bullets. Each bullet leads with the action the owner should take, then
   the reason, then the one number that justifies it. After the bullets, one closing line naming
   the single costliest problem and its number. Nothing else. No headers, no preamble.
5. AUDIENCE FIT - the reader owns the store and is not technical. Ban these words and any
   variant: SLA, GMV, gross merchandise value, sell-through, throughput, churn, blended,
   reconciliation, cycle, variance, YoY, QoQ. Use plain equivalents.

SOURCE MEMO
{memo}

DRAFT SUMMARY
{draft}

OUTPUT FORMAT - use these two headers exactly:

CRITIQUE:
One line per criterion, numbered 1 to 5, in the form:
  <n>. PASS or FAIL - <what specifically> - "<quote the exact offending text, or n/a>"
For criterion 3 give the actual word count. For criterion 2 list every number you could not find
in the memo.

REVISED SUMMARY:
The rewritten summary meeting all five criteria. Nothing after it."""

reflection = ask(
    PROMPT_REFLECT.format(memo=MEMO, draft=summary_before),
    system=REFLECT_SYSTEM,
)
print(reflection)

In [ ]:
parts = re.split(r'REVISED SUMMARY:\s*', reflection, maxsplit=1)
assert len(parts) == 2, 'header not found - check the model followed the output format'
critique, summary_after = parts[0].replace('CRITIQUE:', '').strip(), parts[1].strip()

print(summary_after)
print('\n---')
print('word count:', len(summary_after.split()))

In [ ]:
JARGON = ['sla', 'gmv', 'gross merchandise', 'sell-through', 'sell through', 'throughput',
          'churn', 'blended', 'reconciliation', 'reconcile', 'qoq', 'yoy', 'variance']

NUM = re.compile(r'\d[\d,]*(?:\.\d+)?')

def numbers_in(text):
    return {m.group().replace(',', '').rstrip('.') for m in NUM.finditer(text)}

MEMO_NUMBERS = numbers_in(MEMO)

def check(summary, label):
    words = len(summary.split())
    bullets = len([l for l in summary.splitlines()
                   if l.strip().startswith(('-', '*', '\u2022')) or re.match(r'^\s*\d[\.\)]', l)])
    jargon_hits = sorted({j for j in JARGON if j in summary.lower()})
    unsourced = sorted(numbers_in(summary) - MEMO_NUMBERS)
    return {
        'version': label,
        'words (<=120)': words,
        'bullets (==3)': bullets,
        'jargon hits (==0)': len(jargon_hits),
        'jargon words': ', '.join(jargon_hits) or 'none',
        'numbers not in memo (==0)': len(unsourced),
        'which numbers': ', '.join(unsourced) or 'none',
    }

rows = [check(summary_before, 'BEFORE'), check(summary_after, 'AFTER')]

keys = list(rows[0].keys())[1:]
w = max(len(k) for k in keys) + 2
print(f"{'CHECK':<{w}}{'BEFORE':>12}{'AFTER':>12}")
print('-' * (w + 24))
for k in keys:
    print(f'{k:<{w}}{str(rows[0][k]):>12}{str(rows[1][k]):>12}')

In [ ]:
# Any number in a summary that is not verbatim in the memo is a flag for a human to check,
# not automatic proof of error - a correctly copied number written a different way lands here too.
for r in rows:
    print(f"{r['version']}: {r['which numbers']}")

In [ ]:
GRADER_SYSTEM = 'You are a grader. You output JSON only. You do not explain outside the JSON.'

GRADER = """Score both summaries 1 to 5 on each criterion. 5 is full marks, 1 is a total miss.
Judge only against the memo and the criteria. The order they appear in means nothing.

CRITERIA - the three standard AI evaluation metrics:
relevance   - how closely the summary serves what was asked: what the store owner must decide
coherence   - logical flow and internal consistency
accuracy    - factually correct against the memo, with no figure the memo does not state

CRITERIA - the three task constraints:
length      - 120 words or fewer
format      - exactly 3 action-first bullets plus one closing line naming the costliest problem
audience    - plain language for a non-technical store owner, no operations jargon

MEMO
{memo}

SUMMARY_X
{a}

SUMMARY_Y
{b}

Return ONLY:
{{"X": {{"relevance": n, "coherence": n, "accuracy": n, "length": n, "format": n, "audience": n,
        "note": "under 20 words"}},
 "Y": {{"relevance": n, "coherence": n, "accuracy": n, "length": n, "format": n, "audience": n,
        "note": "under 20 words"}}}}"""

def grade(a, b):
    return json.loads(ask(GRADER.format(memo=MEMO, a=a, b=b), system=GRADER_SYSTEM, json_mode=True))

CRIT = ['relevance', 'coherence', 'accuracy', 'length', 'format', 'audience']

def score_table(scores, col_a='BEFORE', col_b='AFTER'):
    print(f"{'CRITERION':<12}{col_a:>9}{col_b:>9}{'DELTA':>8}")
    print('-' * 38)
    for c in CRIT:
        x, y = scores['X'][c], scores['Y'][c]
        mark = '  <- module metric' if c in ('relevance', 'coherence', 'accuracy') else ''
        print(f'{c:<12}{x:>9}{y:>9}{y - x:>+8}{mark}')
    print('-' * 38)
    tx, ty = sum(scores['X'][c] for c in CRIT), sum(scores['Y'][c] for c in CRIT)
    print(f"{'TOTAL /30':<12}{tx:>9}{ty:>9}{ty - tx:>+8}")
    return tx, ty

scores = grade(summary_before, summary_after)
total_before, total_after = score_table(scores)
print('\nBEFORE note:', scores['X']['note'])
print('AFTER note: ', scores['Y']['note'])

In [ ]:
reflection_2 = ask(
    PROMPT_REFLECT.format(memo=MEMO, draft=summary_after),
    system=REFLECT_SYSTEM,
)
parts_2 = re.split(r'REVISED SUMMARY:\s*', reflection_2, maxsplit=1)
assert len(parts_2) == 2, 'header not found on pass 2'
critique_2, summary_pass2 = parts_2[0].replace('CRITIQUE:', '').strip(), parts_2[1].strip()

print('PASS 2 CRITIQUE')
print('-' * 70)
print(critique_2)
print()
print('PASS 2 SUMMARY')
print('-' * 70)
print(summary_pass2)

In [ ]:
# Same mechanical checks, all three versions.
rows3 = [check(summary_before, 'BEFORE'), check(summary_after, 'PASS 1'), check(summary_pass2, 'PASS 2')]
keys = list(rows3[0].keys())[1:]
w = max(len(k) for k in keys) + 2
print(f"{'CHECK':<{w}}{'BEFORE':>12}{'PASS 1':>12}{'PASS 2':>12}")
print('-' * (w + 36))
for k in keys:
    print(f'{k:<{w}}' + ''.join(f'{str(r[k]):>12}' for r in rows3))

print()
scores_2 = grade(summary_after, summary_pass2)
t1, t2 = score_table(scores_2, col_a='PASS 1', col_b='PASS 2')
print()
gain_1 = total_after - total_before
gain_2 = t2 - t1
print(f'graded gain, pass 1: {gain_1:+d} points')
print(f'graded gain, pass 2: {gain_2:+d} points')
print('Recursion is still paying.' if gain_2 >= gain_1 and gain_2 > 0
      else 'Diminishing returns: pass 1 did the work, pass 2 mostly reshuffles wording.')

In [ ]:
def block(title, text):
    print('=' * 74)
    print(title)
    print('=' * 74)
    print(text)
    print()

block('BEFORE - loose prompt', summary_before)
block('THE CRITIQUE', critique)
block('AFTER - revised against the five criteria', summary_after)
block('PASS 2 - RSIP, a second reflection on the revision', summary_pass2)